In [ ]:
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

# 创建环境
env = gym.make('ALE/Breakout-v5', render_mode="human")
obs, info = env.reset()

# 运行示例
for step in range(1000):
    action = env.action_space.sample()  # 随机动作
    obs, reward, terminated, truncated, info = env.step(action)
    
    if terminated or truncated:
        obs, info = env.reset()
    
    if step % 100 == 0:
        print(f"Step {step}, Reward: {reward}")

env.close()

In [ ]:
env.observation_spec() 

In [ ]:
env.action_spec() 

In [ ]:
env.time_step_spec() 

In [ ]:
env.gym.get_action_meanings() 

In [ ]:
from tf_agents.environments.wrappers import ActionRepeat 
repeating_env = ActionRepeat(env, times=4) 
from gym.wrappers import TimeLimit 
limited_repeating_env = suite_gym.load( 
    "Breakout-v4", 
    gym_env_wrappers=[lambda env: TimeLimit(env, max_episode_steps=10000)], 
    env_wrappers=[lambda env: ActionRepeat(env, times=4)])

from tf_agents.environments import suite_atari 
from tf_agents.environments.atari_preprocessing import AtariPreprocessing 
from tf_agents.environments.atari_wrappers import FrameStack4 
max_episode_steps = 27000 # <=> 108k ALE frames since 1 step = 4 frames 
environment_name = "BreakoutNoFrameskip-v4" 
env = suite_atari.load( 
    environment_name, 
    max_episode_steps=max_episode_steps, 
    gym_env_wrappers=[AtariPreprocessing, FrameStack4]) 

from tf_agents.environments.tf_py_environment import TFPyEnvironment 
tf_env = TFPyEnvironment(env) 


In [ ]:
from tf_agents.networks.q_network import QNetwork 
preprocessing_layer = keras.layers.Lambda( 
                         lambda obs: tf.cast(obs, np.float32) / 255.) 
conv_layer_params=[(32, (8, 8), 4), (64, (4, 4), 2), (64, (3, 3), 1)] 
fc_layer_params=[512] 
q_net = QNetwork( 
    tf_env.observation_spec(), 
    tf_env.action_spec(), 
    preprocessing_layers=preprocessing_layer, 
    conv_layer_params=conv_layer_params, 
    fc_layer_params=fc_layer_params)

In [ ]:
from tf_agents.agents.dqn.dqn_agent import DqnAgent 
train_step = tf.Variable(0) 
update_period = 4 # train the model every 4 steps 
optimizer = keras.optimizers.RMSprop(lr=2.5e-4, rho=0.95, momentum=0.0, 
                                    epsilon=0.00001, centered=True) 
epsilon_fn = keras.optimizers.schedules.PolynomialDecay( 
    initial_learning_rate=1.0, # initial ε 
    decay_steps=250000 // update_period, # <=> 1,000,000 ALE frames 
    end_learning_rate=0.01) # final ε 
agent = DqnAgent(tf_env.time_step_spec(), 
                 tf_env.action_spec(), 
                 q_network=q_net, 
                 optimizer=optimizer, 
                 target_update_period=2000, # <=> 32,000 ALE frames 
                 td_errors_loss_fn=keras.losses.Huber(reduction="none"), 
                 gamma=0.99, # discount factor 
                 train_step_counter=train_step, 
                 epsilon_greedy=lambda: epsilon_fn(train_step)) 
agent.initialize()

from tf_agents.replay_buffers import tf_uniform_replay_buffer 
replay_buffer = tf_uniform_replay_buffer.TFniformReplayBuffer( 
    data_spec=agent.collect_data_spec, 
    batch_size=tf_env.batch_size, 
    max_length=1000000) 
replay_buffer_observer = replay_buffer.add_batch 

In [ ]:
class ShowProgress: 
    def __init__(self, total): 
        self.counter = 0 
        self.total = total 
    def __call__(self, trajectory): 
        if not trajectory.is_boundary(): 
            self.counter += 1 
        if self.counter % 100 == 0: 
            print("\r{}/{}".format(self.counter, self.total), end="")

from tf_agents.metrics import tf_metrics 
train_metrics = [ 
    tf_metrics.NumberOfEpisodes(), 
    tf_metrics.EnvironmentSteps(), 
    tf_metrics.AverageReturnMetric(), 
    tf_metrics.AverageEpisodeLengthMetric(), 
]

In [ ]:
from tf_agents.eval.metric_utils import log_metrics 
import logging 
logging.get_logger().set_level(logging.INFO) 
log_metrics(train_metrics) 

In [ ]:
from tf_agents.drivers.dynamic_step_driver import DynamicStepDriver 
collect_driver = DynamicStepDriver( 
    tf_env, 
    agent.collect_policy, 
    observers=[replay_buffer_observer] + training_metrics, 
    num_steps=update_period) # collect 4 steps for each training iteration 

In [ ]:
from tf_agents.policies.random_tf_policy import RandomTFPolicy 
initial_collect_policy = RandomTFPolicy(tf_env.time_step_spec(), 
                                        tf_env.action_spec()) 
init_driver = DynamicStepDriver( 
    tf_env, 
    initial_collect_policy, 
    observers=[replay_buffer.add_batch, ShowProgress(20000)], 
    num_steps=20000) # <=> 80,000 ALE frames 
final_time_step, final_policy_state = init_driver.run()

trajectories, buffer_info = replay_buffer.get_next(sample_batch_size=2, num_steps=3)

dataset = replay_buffer.as_dataset( 
    sample_batch_size=64, 
    num_steps=2, 
    num_parallel_calls=3).prefetch(3)

In [ ]:
from tf_agents.utils.common import function 
collect_driver.run = function(collect_driver.run) 
agent.train = function(agent.train) 
def train_agent(n_iterations): 
    time_step = None 
    policy_state = agent.collect_policy.get_initial_state(tf_env.batch_size) 
    iterator = iter(dataset) 
    for iteration in range(n_iterations): 
        time_step, policy_state = collect_driver.run(time_step, policy_state) 
        trajectories, buffer_info = next(iterator) 
        train_loss = agent.train(trajectories) 
        print("\r{} loss:{:.5f}".format( 
            iteration, train_loss.loss.numpy()), end="") 
        if iteration % 1000 == 0: 
            log_metrics(train_metrics)

train_agent(10000000)